# Task MP-DATA-004: Eligibility Grids, Target Availability, & Corroboration Gaps
**Repository:** `ankit-choubey/Mandi_Prediction`  
**Execution Environment:** Kaggle CPU Kernel (Internet Disabled)  
**Evaluated Dataset:** `theankitchoubey/mandi-prediction-audit-003b` (Version 1)  

### Objectives
1. **Unmatched Excel Rows:** Classify the 2,139 Excel rows inside the panel window that did not match the validated panel into explicit categories: quarantined conflicts, low-observation drops, raw spelling differences, absent from raw files, or other. Export `unmatched_excel_sample.csv`.
2. **The 77 Price Differences:** Profile size distribution, test whether Excel modal price equals any quarantined conflicting quote on that natural key, and record dates involved. Export `diff_77_analysis.json`.
3. **Quarantine Adjudication Test:** Test simple deterministic adjudication rules (`first`, `last`, `min`, `max`, `mean`, `median`) against external reference quotes from Excel and CEDA. Report concordance rates. Export `quarantine_adjudication_summary.json`.
4. **Target Availability:** Measure forward observation existence at $T+7$, $T+6..T+8$, $T+5..T+9$, and $t_{next} \le T+10$ across all 1,297 retained series. Report day-of-week and gap patterns. Export `target_availability_summary.json` and `target_availability_breakdown.csv`.
5. **Eligibility Grids:** Evaluate counts of qualifying series and observations across a 90-cell grid of $(N_{obs}, \text{cov}, \text{gap}_{max}, \text{recency})$. Break out the six focus crops (Wheat, Potato, Onion, Tomato, Paddy, Rice) by district and market. Export `eligibility_grids.csv` and `focus_crops_eligibility.csv`.
6. **Chronological Split Feasibility:** Evaluate origin counts in the final 15% evaluation period after an 8-day embargo, reporting series with $\ge 20$ and $\ge 50$ test origins. Export `chronological_split_feasibility.json`.
7. **Environment & Input Verification:** Record library versions, seeds, and SHA-256 hashes of all 30 input files.


In [ ]:
# Cell 1: Environment, Library Versions, Random Seeds, and Input Discovery
import os
import sys
import glob
import json
import hashlib
import time
import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import sklearn
import openpyxl

print("=== ENVIRONMENT & LIBRARY VERSIONS ===")
print(f"Python: {sys.version}")
print(f"Platform: {sys.platform}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"sklearn: {sklearn.__version__}")
print(f"openpyxl: {openpyxl.__version__}")

# Random seed
np.random.seed(42)
print("Random seed initialized to 42.")

# Locate dataset root
KAGGLE_INPUT_DIR = None
search_candidates = [
    "/kaggle/input/mandi-prediction-audit-003b",
    "/kaggle/input/datasets/theankitchoubey/mandi-prediction-audit-003b",
    "/kaggle/input",
    "."
]
for candidate in search_candidates:
    if os.path.exists(candidate):
        if os.path.exists(os.path.join(candidate, "5 year dataset")):
            KAGGLE_INPUT_DIR = candidate
            break
        elif os.path.exists(os.path.join(candidate, "mandi-prediction-audit-003b", "5 year dataset")):
            KAGGLE_INPUT_DIR = os.path.join(candidate, "mandi-prediction-audit-003b")
            break

if KAGGLE_INPUT_DIR is None:
    matches = glob.glob("/kaggle/input/**/GHAZIABAD.csv", recursive=True) or glob.glob("**/GHAZIABAD.csv", recursive=True)
    if matches:
        KAGGLE_INPUT_DIR = str(Path(matches[0]).parent.parent)

print(f"Resolved Dataset Payload Root: {KAGGLE_INPUT_DIR}")
if KAGGLE_INPUT_DIR not in sys.path:
    sys.path.insert(0, KAGGLE_INPUT_DIR)
src_path = os.path.join(KAGGLE_INPUT_DIR, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

def compute_sha256(filepath):
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    return hasher.hexdigest()

print("\n=== INPUT FILES & SHA-256 MANIFEST ===")
input_file_records = []
for root, dirs, files in os.walk(KAGGLE_INPUT_DIR):
    for f in sorted(files):
        if f.endswith((".csv", ".json", ".py", ".xlsx", ".txt")):
            full_p = os.path.join(root, f)
            rel_p = os.path.relpath(full_p, KAGGLE_INPUT_DIR)
            sz = os.path.getsize(full_p)
            sha = compute_sha256(full_p)
            input_file_records.append({"file": rel_p, "size_bytes": sz, "sha256": sha})
            print(f"{sz:>10} bytes | {sha[:16]}... | {rel_p}")

print(f"Total verified input files: {len(input_file_records)}")


In [ ]:
# Cell 2: Part 1 - Unmatched Excel Rows Analysis & Classification
from src.ingest_agmarknet_arrivals import load_arrivals_file
from src.data_pipeline import MandiDataPipeline, DISTRICT_NORMALIZATION

# 1. Load validated panel
val_csv_path = os.path.join(KAGGLE_INPUT_DIR, "data/processed/mandi_5year_validated.csv")
val_df = pd.read_csv(val_csv_path)
val_df["Arrival_Date"] = pd.to_datetime(val_df["Arrival_Date"]).dt.strftime("%Y-%m-%d")
val_df["Modal_Price_Clean"] = pd.to_numeric(val_df["Modal_Price"], errors="coerce")
print(f"Loaded validated panel: {len(val_df):,} rows")

# 2. Load Excel files
excel_files = sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "raw_downloads/*.xlsx")))
if not excel_files:
    excel_files = sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "data/raw/arrivals/historical/raw_downloads/*.xlsx")))
if not excel_files:
    excel_files = sorted(glob.glob(f"{KAGGLE_INPUT_DIR}/**/raw_downloads/*.xlsx", recursive=True))
if not excel_files:
    excel_files = sorted(glob.glob(f"{KAGGLE_INPUT_DIR}/**/*.xlsx", recursive=True))

print(f"Discovered {len(excel_files)} Excel files.")
excel_dfs = [load_arrivals_file(ef) for ef in excel_files]
df_excel = pd.concat(excel_dfs, ignore_index=True)
df_excel["Modal_Price_Clean"] = pd.to_numeric(df_excel["Modal_Price"], errors="coerce")
print(f"Total Agmarknet Excel records: {len(df_excel):,}")

# Filter to panel date window (<= 2026-09-30)
excel_in_window = df_excel[df_excel["Arrival_Date"] <= "2026-09-30"].copy()
print(f"Excel records inside panel window: {len(excel_in_window):,}")

# 3. Match natural key with validated panel
nat_key = ["District_Std", "Market", "Commodity_Std", "Variety", "Grade", "Arrival_Date"]
panel_key = ["District", "Market", "Commodity", "Variety", "Grade", "Arrival_Date"]

val_keys = set(tuple(x) for x in val_df[panel_key].values)
excel_in_window["is_matched"] = [tuple(x) in val_keys for x in excel_in_window[nat_key].values]

matched_excel = excel_in_window[excel_in_window["is_matched"]].copy()
unmatched_excel = excel_in_window[~excel_in_window["is_matched"]].copy()

print(f"Matched Excel rows: {len(matched_excel):,}")
print(f"Unmatched Excel rows: {len(unmatched_excel):,}")

# 4. Load Quarantined Conflicts
q_conf_path = os.path.join(KAGGLE_INPUT_DIR, "data/quarantine/quarantined_conflicting_records.csv")
df_q_conf = pd.read_csv(q_conf_path)
df_q_conf["arrival_date"] = pd.to_datetime(df_q_conf["arrival_date"]).dt.strftime("%Y-%m-%d")
conf_keys = set(tuple(x) for x in df_q_conf[["district", "market", "commodity", "variety", "grade", "arrival_date"]].values)
print(f"Loaded quarantined conflicting records: {len(df_q_conf):,} rows, {len(conf_keys):,} distinct keys")

# 5. Identify Low-Observation Dropped Series (< 20 obs) from raw 5-year pipeline
five_year_dir = os.path.join(KAGGLE_INPUT_DIR, "5 year dataset")
pipeline = MandiDataPipeline(input_dir=five_year_dir, min_series_records=20)
source_files = pipeline.find_source_files()

raw_dfs = []
for sf in source_files:
    df_clean, _ = pipeline.clean_single_file(sf)
    raw_dfs.append(df_clean)
raw_all = pd.concat(raw_dfs, ignore_index=True)
raw_all["Arrival_Date"] = pd.to_datetime(raw_all["Arrival_Date"]).dt.strftime("%Y-%m-%d")

# Exclude conflicts
key_grade_date = ["District", "Market", "Commodity", "Variety", "Grade", "Arrival_Date"]
is_conf = raw_all.set_index(key_grade_date).index.isin(conf_keys)
surv_pre_filter = raw_all[~is_conf].drop_duplicates(subset=key_grade_date, keep="first").copy()

# Series counts
series_cols = ["District", "Market", "Commodity", "Variety", "Grade"]
surv_pre_filter["series_id"] = surv_pre_filter[series_cols].astype(str).agg("__".join, axis=1)
counts = surv_pre_filter["series_id"].value_counts()
low_obs_series_ids = set(counts[counts < 20].index)
print(f"Identified {len(low_obs_series_ids)} low-observation series (<20 records)")

# Sets for raw matching
low_obs_keys = set(tuple(x) for x in surv_pre_filter[surv_pre_filter["series_id"].isin(low_obs_series_ids)][key_grade_date].values)
raw_dc_dates = set(zip(raw_all["District"], raw_all["Commodity"], raw_all["Arrival_Date"]))
raw_keys = set(tuple(x) for x in raw_all[key_grade_date].values)

# 6. Classify Unmatched Rows
classifications = []
for _, row in unmatched_excel.iterrows():
    k = (row["District_Std"], row["Market"], row["Commodity_Std"], row["Variety"], row["Grade"], row["Arrival_Date"])
    dc_date = (row["District_Std"], row["Commodity_Std"], row["Arrival_Date"])
    
    if k in conf_keys:
        cls = "key_in_quarantined_conflicts"
    elif k in low_obs_keys or "__".join([str(row["District_Std"]), str(row["Market"]), str(row["Commodity_Std"]), str(row["Variety"]), str(row["Grade"])]) in low_obs_series_ids:
        cls = "key_in_low_obs_drops"
    elif dc_date in raw_dc_dates:
        cls = "key_in_raw_diff_spelling"
    elif dc_date not in raw_dc_dates:
        cls = "absent_from_raw_files"
    else:
        cls = "other"
    classifications.append(cls)

unmatched_excel["unmatched_category"] = classifications

print("\n=== UNMATCHED EXCEL ROWS CLASSIFICATION SUMMARY ===")
cat_counts = unmatched_excel["unmatched_category"].value_counts()
print(cat_counts)

print("\n=== CLASSIFICATION BREAKDOWN BY DISTRICT & COMMODITY ===")
dist_crop_breakdown = unmatched_excel.groupby(["District_Std", "Commodity_Std", "unmatched_category"]).size().unstack(fill_value=0)
print(dist_crop_breakdown.to_string())

# Export sample (up to 2,000 rows)
sample_cols = ["District_Std", "Market", "Commodity_Std", "Variety", "Grade", "Arrival_Date", "Modal_Price_Clean", "unmatched_category"]
unmatched_sample = unmatched_excel[sample_cols].head(2000)
unmatched_sample.to_csv("unmatched_excel_sample.csv", index=False)
print(f"Exported unmatched_excel_sample.csv ({len(unmatched_sample):,} rows)")

unmatched_summary = {
    "total_excel_rows_in_window": len(excel_in_window),
    "matched_rows": len(matched_excel),
    "unmatched_rows": len(unmatched_excel),
    "category_counts": cat_counts.to_dict(),
    "breakdown_by_district_crop": {f"{k[0]}__{k[1]}__{k[2]}": int(v) for k, v in unmatched_excel.groupby(["District_Std", "Commodity_Std", "unmatched_category"]).size().items()}
}
with open("unmatched_excel_summary.json", "w") as f:
    json.dump(unmatched_summary, f, indent=2)



In [ ]:
# Cell 3: Part 2 - Detailed Analysis of the 77 Price Differences
merged_excel = pd.merge(
    matched_excel,
    val_df,
    left_on=["District_Std", "Market", "Commodity_Std", "Variety", "Grade", "Arrival_Date"],
    right_on=["District", "Market", "Commodity", "Variety", "Grade", "Arrival_Date"],
    how="inner",
    suffixes=("_excel", "_panel")
)
merged_excel["price_diff"] = merged_excel["Modal_Price_Clean_excel"] - merged_excel["Modal_Price_Clean_panel"]
diff_rows = merged_excel[merged_excel["price_diff"] != 0].copy()

print(f"Total non-zero price difference rows: {len(diff_rows)}")

# 1. Size distribution
diff_stats = {
    "count": len(diff_rows),
    "mean_diff": round(float(diff_rows["price_diff"].mean()), 2),
    "std_diff": round(float(diff_rows["price_diff"].std()), 2),
    "min_diff": round(float(diff_rows["price_diff"].min()), 2),
    "p10_diff": round(float(np.percentile(diff_rows["price_diff"], 10)), 2),
    "p25_diff": round(float(np.percentile(diff_rows["price_diff"], 25)), 2),
    "median_diff": round(float(np.percentile(diff_rows["price_diff"], 50)), 2),
    "p75_diff": round(float(np.percentile(diff_rows["price_diff"], 75)), 2),
    "p90_diff": round(float(np.percentile(diff_rows["price_diff"], 90)), 2),
    "max_diff": round(float(diff_rows["price_diff"].max()), 2),
    "mean_abs_diff": round(float(np.abs(diff_rows["price_diff"]).mean()), 2),
    "median_abs_diff": round(float(np.median(np.abs(diff_rows["price_diff"]))), 2)
}
print("Price Differences Size Distribution:")
for k, v in diff_stats.items():
    print(f"  {k}: {v}")

# 2. Check if Excel price equals any quarantined conflicting price
equals_quarantine_count = 0
conf_price_matches = []
for _, r in diff_rows.iterrows():
    d, m, c, v, g, dt = r["District_Std"], r["Market"], r["Commodity_Std"], r["Variety"], r["Grade"], r["Arrival_Date"]
    pe = r["Modal_Price_Clean_excel"]
    # Check quarantine match
    q_matches = df_q_conf[
        (df_q_conf["district"] == d) & 
        (df_q_conf["market"] == m) & 
        (df_q_conf["commodity"] == c) & 
        (df_q_conf["variety"] == v) & 
        (df_q_conf["grade"] == g) & 
        (df_q_conf["arrival_date"] == dt)
    ]
    matched_q_price = False
    if len(q_matches) > 0:
        q_prices = q_matches["modal_price"].tolist()
        if pe in q_prices:
            matched_q_price = True
            equals_quarantine_count += 1
    conf_price_matches.append({
        "district": d, "market": m, "commodity": c, "variety": v, "grade": g, "date": dt,
        "excel_price": pe, "panel_price": r["Modal_Price_Clean_panel"], "diff": r["price_diff"],
        "found_in_quarantine": len(q_matches) > 0,
        "equals_quarantined_price": matched_q_price
    })

print(f"\nExcel prices exactly matching a quarantined conflicting price on same natural key: {equals_quarantine_count} / {len(diff_rows)}")

# 3. Dates involved
dates_involved = sorted(diff_rows["Arrival_Date"].unique().tolist())
print(f"Dates involved ({len(dates_involved)} distinct dates): {dates_involved}")

diff_analysis = {
    "diff_count": len(diff_rows),
    "size_distribution": diff_stats,
    "equals_quarantined_price_count": equals_quarantine_count,
    "distinct_dates_count": len(dates_involved),
    "dates_involved": dates_involved,
    "detailed_records": conf_price_matches
}
with open("diff_77_analysis.json", "w") as f:
    json.dump(diff_analysis, f, indent=2)
print("Saved diff_77_analysis.json")



In [ ]:
# Cell 4: Part 3 - Quarantine Adjudication Test (Report Only, No Rule Change)
# Reference prices from Excel and CEDA
# 1. Load CEDA Price Data
ceda_csv_path = os.path.join(KAGGLE_INPUT_DIR, "raw_downloads/ceda price data.csv")
if not os.path.exists(ceda_csv_path):
    ceda_candidates = [
        os.path.join(KAGGLE_INPUT_DIR, "data/raw/arrivals/historical/raw_downloads/ceda price data.csv"),
        os.path.join(KAGGLE_INPUT_DIR, "data/raw/arrivals/historical/ceda price data.csv")
    ]
    for cand in ceda_candidates:
        if os.path.exists(cand):
            ceda_csv_path = cand
            break
    else:
        matches = glob.glob(f"{KAGGLE_INPUT_DIR}/**/ceda price data.csv", recursive=True)
        if matches:
            ceda_csv_path = matches[0]

df_ceda = pd.read_csv(ceda_csv_path)
df_ceda["t"] = pd.to_datetime(df_ceda["t"]).dt.strftime("%Y-%m-%d")

# Build reference price lookups
ref_lookup = {}
# From Excel
for _, r in df_excel.iterrows():
    if pd.notna(r["Modal_Price_Clean"]):
        k = (r["District_Std"], r["Market"], r["Commodity_Std"], r["Variety"], r["Grade"], r["Arrival_Date"])
        ref_lookup[k] = ("excel", float(r["Modal_Price_Clean"]))

# From CEDA (Lucknow Wheat)
for _, r in df_ceda.iterrows():
    k = (r["district_name"], r["market_name"], r["cmdty"], r["variety"], "FAQ", r["t"])
    if k not in ref_lookup:
        ref_lookup[k] = ("ceda", float(r["p_modal"]))

print(f"Constructed reference price lookup with {len(ref_lookup):,} entries.")

# Group quarantine records by natural key
conf_key_cols = ["district", "market", "commodity", "variety", "grade", "arrival_date"]
quarantine_groups = df_q_conf.groupby(conf_key_cols)
print(f"Total conflicting quarantine groups: {len(quarantine_groups):,}")

rule_counts = {
    "first": 0,
    "last": 0,
    "min": 0,
    "max": 0,
    "mean": 0,
    "median": 0
}
evaluated_groups = 0
evaluated_details = []

for key, grp in quarantine_groups:
    ref_match = ref_lookup.get(key)
    if not ref_match:
        continue
    source_type, ref_p = ref_match
    prices = grp["modal_price"].dropna().tolist()
    if len(prices) == 0:
        continue
    
    evaluated_groups += 1
    
    cand_first = float(prices[0])
    cand_last = float(prices[-1])
    cand_min = float(min(prices))
    cand_max = float(max(prices))
    cand_mean = round(float(np.mean(prices)))
    cand_median = float(np.median(prices))
    
    m_first = (cand_first == ref_p)
    m_last = (cand_last == ref_p)
    m_min = (cand_min == ref_p)
    m_max = (cand_max == ref_p)
    m_mean = (cand_mean == ref_p)
    m_median = (cand_median == ref_p)
    
    if m_first: rule_counts["first"] += 1
    if m_last: rule_counts["last"] += 1
    if m_min: rule_counts["min"] += 1
    if m_max: rule_counts["max"] += 1
    if m_mean: rule_counts["mean"] += 1
    if m_median: rule_counts["median"] += 1
    
    evaluated_details.append({
        "key": list(key),
        "source": source_type,
        "ref_price": ref_p,
        "conflicting_prices": prices,
        "matches": {
            "first": m_first, "last": m_last, "min": m_min, "max": m_max, "mean": m_mean, "median": m_median
        }
    })

print(f"\nEvaluated {evaluated_groups} conflicting groups that overlap external reference data.")
adjudication_summary = {
    "total_quarantine_groups": len(quarantine_groups),
    "overlapping_evaluated_groups": evaluated_groups,
    "rule_concordance_counts": rule_counts,
    "rule_concordance_pcts": {
        k: round(float(v / evaluated_groups * 100), 2) if evaluated_groups > 0 else 0.0
        for k, v in rule_counts.items()
    }
}
print("Adjudication Rule Concordance Rates:")
for k, pct in adjudication_summary["rule_concordance_pcts"].items():
    print(f"  Rule '{k}': {rule_counts[k]}/{evaluated_groups} ({pct}%)")

with open("quarantine_adjudication_summary.json", "w") as f:
    json.dump({"summary": adjudication_summary, "details": evaluated_details}, f, indent=2)
print("Saved quarantine_adjudication_summary.json")



In [ ]:
# Cell 5: Part 4 - Target Availability & Forward Gap Analysis
# Parse arrival date as datetime
val_df["Arrival_Date_dt"] = pd.to_datetime(val_df["Arrival_Date"])
val_df["day_of_week"] = val_df["Arrival_Date_dt"].dt.day_name()
val_df["day_of_week_num"] = val_df["Arrival_Date_dt"].dt.dayofweek # 0=Mon, 6=Sun

print("Computing forward target availability for all 1,297 retained series...")
target_records = []
series_grouped = val_df.groupby("series_id")

all_gaps = []

for sid, grp in series_grouped:
    dist = grp["District"].iloc[0]
    comm = grp["Commodity"].iloc[0]
    mkt = grp["Market"].iloc[0]
    
    # Sort distinct dates
    dates = sorted(grp["Arrival_Date_dt"].drop_duplicates().tolist())
    date_set = set(dates)
    
    # Compute gaps between consecutive observations
    for i in range(len(dates) - 1):
        gap = (dates[i+1] - dates[i]).days
        all_gaps.append(gap)
    
    for i, t in enumerate(dates):
        # Target availability checks
        t_plus_7 = t + datetime.timedelta(days=7)
        has_t7 = (t_plus_7 in date_set)
        
        has_t6_8 = any((t + datetime.timedelta(days=k)) in date_set for k in [6, 7, 8])
        has_t5_9 = any((t + datetime.timedelta(days=k)) in date_set for k in [5, 6, 7, 8, 9])
        
        next_d = dates[i+1] if i + 1 < len(dates) else None
        next_within_10 = ((next_d - t).days <= 10) if next_d else False
        
        target_records.append({
            "series_id": sid,
            "district": dist,
            "commodity": comm,
            "market": mkt,
            "date": t.strftime("%Y-%m-%d"),
            "dow_num": t.dayofweek,
            "dow_name": t.strftime("%A"),
            "has_t7": has_t7,
            "has_t6_8": has_t6_8,
            "has_t5_9": has_t5_9,
            "next_within_10": next_within_10
        })

df_targets = pd.DataFrame(target_records)
print(f"Total observation origins evaluated: {len(df_targets):,}")

# Overall panel target availability
overall_shares = {
    "total_origins": len(df_targets),
    "share_has_t7": round(float(df_targets["has_t7"].mean()), 4),
    "share_has_t6_8": round(float(df_targets["has_t6_8"].mean()), 4),
    "share_has_t5_9": round(float(df_targets["has_t5_9"].mean()), 4),
    "share_next_within_10": round(float(df_targets["next_within_10"].mean()), 4)
}
print("\n=== OVERALL TARGET AVAILABILITY SHARES ===")
for k, v in overall_shares.items():
    print(f"  {k}: {v}")

# Day-of-week breakdown
all_dows = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_raw = df_targets.groupby("dow_name").agg(
    origins=("has_t7", "count"),
    share_t7=("has_t7", "mean"),
    share_t6_8=("has_t6_8", "mean"),
    share_t5_9=("has_t5_9", "mean"),
    share_next_within_10=("next_within_10", "mean")
)
dow_breakdown = dow_raw.reindex(all_dows).fillna(0)
print("\n=== DAY-OF-WEEK DISTRIBUTION & TARGET AVAILABILITY ===")
print(dow_breakdown.to_string())

# Gap distribution
gaps_s = pd.Series(all_gaps)
gap_dist = {
    "count": len(gaps_s),
    "mean_gap": round(float(gaps_s.mean()), 2),
    "median_gap": round(float(gaps_s.median()), 2),
    "p90_gap": round(float(np.percentile(gaps_s, 90)), 2),
    "max_gap": round(float(gaps_s.max()), 2),
    "share_gap_1_day": round(float((gaps_s == 1).mean()), 4),
    "share_gap_2_days": round(float((gaps_s == 2).mean()), 4),
    "share_gap_3_plus_days": round(float((gaps_s >= 3).mean()), 4),
    "share_gap_gt_7_days": round(float((gaps_s > 7).mean()), 4)
}
print("\n=== OBSERVATION GAP PATTERNS ===")
for k, v in gap_dist.items():
    print(f"  {k}: {v}")

# Commodity & District Breakdown
comm_dist_breakdown = df_targets.groupby(["district", "commodity"]).agg(
    total_origins=("has_t7", "count"),
    share_t7=("has_t7", "mean"),
    share_t6_8=("has_t6_8", "mean"),
    share_t5_9=("has_t5_9", "mean"),
    share_next_within_10=("next_within_10", "mean")
).reset_index()

# Export breakdown CSV
comm_dist_breakdown.to_csv("target_availability_breakdown.csv", index=False)
print("Saved target_availability_breakdown.csv")

target_summary = {
    "overall_shares": overall_shares,
    "day_of_week_breakdown": dow_breakdown.to_dict(orient="index"),
    "gap_distribution": gap_dist,
    "top_commodities_breakdown": comm_dist_breakdown.head(30).to_dict(orient="records")
}
with open("target_availability_summary.json", "w") as f:
    json.dump(target_summary, f, indent=2)
print("Saved target_availability_summary.json")



In [ ]:
# Cell 6: Part 5 - Eligibility Grids (90 Combinations & Focus Crop Breakout)
# Profile series attributes
series_metrics = []
for sid, grp in series_grouped:
    dates = sorted(grp["Arrival_Date_dt"].drop_duplicates().tolist())
    n_obs = len(grp)
    first_dt = dates[0]
    last_dt = dates[-1]
    span = (last_dt - first_dt).days + 1
    cov = n_obs / span if span > 0 else 0.0
    
    max_gap = 0
    if len(dates) > 1:
        gaps = [(dates[i+1] - dates[i]).days for i in range(len(dates) - 1)]
        max_gap = max(gaps)
    
    series_metrics.append({
        "series_id": sid,
        "district": grp["District"].iloc[0],
        "commodity": grp["Commodity"].iloc[0],
        "market": grp["Market"].iloc[0],
        "variety": grp["Variety"].iloc[0],
        "grade": grp["Grade"].iloc[0],
        "n_obs": n_obs,
        "span_days": span,
        "cov_share": cov,
        "max_gap": max_gap,
        "last_date": last_dt
    })

df_series = pd.DataFrame(series_metrics)
print(f"Profiled {len(df_series):,} series.")

panel_max_date = pd.Timestamp("2026-09-30")
cutoff_30 = panel_max_date - pd.Timedelta(days=30)
cutoff_60 = panel_max_date - pd.Timedelta(days=60)

# Evaluate 90-cell Grid
min_obs_vals = [30, 60, 120, 250, 500]
min_cov_vals = [0.3, 0.5, 0.7]
max_gap_vals = [14, 30, 60]
recency_options = [(30, cutoff_30), (60, cutoff_60)]

grid_records = []
for min_obs in min_obs_vals:
    for min_cov in min_cov_vals:
        for max_gap in max_gap_vals:
            for rec_days, c_date in recency_options:
                cond = (
                    (df_series["n_obs"] >= min_obs) &
                    (df_series["cov_share"] >= min_cov) &
                    (df_series["max_gap"] <= max_gap) &
                    (df_series["last_date"] >= c_date)
                )
                matching = df_series[cond]
                grid_records.append({
                    "min_obs": min_obs,
                    "min_cov": min_cov,
                    "max_gap": max_gap,
                    "recency_days": rec_days,
                    "qualifying_series_count": len(matching),
                    "total_qualifying_observations": int(matching["n_obs"].sum()),
                    "qualifying_series_share": round(len(matching) / len(df_series), 4)
                })

df_grid = pd.DataFrame(grid_records)
df_grid.to_csv("eligibility_grids.csv", index=False)
print(f"Exported eligibility_grids.csv ({len(df_grid)} combinations)")

# Six focus crops breakout: Wheat, Potato, Onion, Tomato, Paddy, Rice
focus_crops = ["Wheat", "Potato", "Onion", "Tomato", "Paddy(Dhan)(Common)", "Rice"]
df_series["is_focus"] = df_series["commodity"].str.contains("Wheat|Potato|Onion|Tomato|Paddy|Rice", case=False, na=False)

focus_breakout = []
tiers = [
    ("Strict", 250, 0.5, 30, 30, cutoff_30),
    ("Balanced", 120, 0.3, 60, 60, cutoff_60),
    ("Inclusive", 60, 0.3, 60, 60, cutoff_60)
]

for tier_name, m_obs, m_cov, m_gap, r_days, c_date in tiers:
    for (dist, mkt, comm), grp in df_series[df_series["is_focus"]].groupby(["district", "market", "commodity"]):
        cond = (
            (grp["n_obs"] >= m_obs) &
            (grp["cov_share"] >= m_cov) &
            (grp["max_gap"] <= m_gap) &
            (grp["last_date"] >= c_date)
        )
        qual_count = int(cond.sum())
        total_count = len(grp)
        focus_breakout.append({
            "tier": tier_name,
            "district": dist,
            "market": mkt,
            "commodity": comm,
            "total_series": total_count,
            "qualifying_series": qual_count,
            "qualifying_share": round(qual_count / total_count, 2)
        })

df_focus = pd.DataFrame(focus_breakout)
df_focus.to_csv("focus_crops_eligibility.csv", index=False)
print(f"Exported focus_crops_eligibility.csv ({len(df_focus)} rows)")

print("\nSample Eligibility Grid Cells:")
print(df_grid.head(10).to_string(index=False))



In [ ]:
# Cell 7: Part 6 - Chronological Train/Test Split Feasibility
# Date span of panel
panel_min_date = df_targets["date"].min()
panel_max_date = df_targets["date"].max()
span_days = (pd.to_datetime(panel_max_date) - pd.to_datetime(panel_min_date)).days + 1

# Final 15% window start date
test_start_dt = pd.to_datetime(panel_min_date) + pd.Timedelta(days=int(span_days * 0.85))
embargo_end_dt = test_start_dt - pd.Timedelta(days=1)
train_end_dt = test_start_dt - pd.Timedelta(days=8)

print(f"Panel Date Span: {panel_min_date} to {panel_max_date} ({span_days} days)")
print(f"Training Period: {panel_min_date} to {train_end_dt.strftime('%Y-%m-%d')}")
print(f"Embargo Window (8 days): {(train_end_dt + pd.Timedelta(days=1)).strftime('%Y-%m-%d')} to {embargo_end_dt.strftime('%Y-%m-%d')}")
print(f"Test Evaluation Window (Final 15%): {test_start_dt.strftime('%Y-%m-%d')} to {panel_max_date}")

# Count test origins per series in test window
df_targets["date_dt"] = pd.to_datetime(df_targets["date"])
df_test_origins = df_targets[df_targets["date_dt"] >= test_start_dt]

series_test_counts = df_test_origins.groupby("series_id").size().to_dict()

# Evaluate across representative tiers from eligibility grids
chronological_feasibility = []
for tier_name, m_obs, m_cov, m_gap, r_days, c_date in tiers:
    cond = (
        (df_series["n_obs"] >= m_obs) &
        (df_series["cov_share"] >= m_cov) &
        (df_series["max_gap"] <= m_gap) &
        (df_series["last_date"] >= c_date)
    )
    qual_series = df_series[cond]
    
    n_qual = len(qual_series)
    test_counts = [series_test_counts.get(sid, 0) for sid in qual_series["series_id"]]
    
    ge_20 = sum(1 for cnt in test_counts if cnt >= 20)
    ge_50 = sum(1 for cnt in test_counts if cnt >= 50)
    
    chronological_feasibility.append({
        "tier": tier_name,
        "criteria": f"obs>={m_obs}, cov>={m_cov}, gap<={m_gap}, recency<={r_days}d",
        "total_qualifying_series": n_qual,
        "series_with_ge_20_test_origins": ge_20,
        "share_ge_20": round(ge_20 / n_qual, 4) if n_qual > 0 else 0.0,
        "series_with_ge_50_test_origins": ge_50,
        "share_ge_50": round(ge_50 / n_qual, 4) if n_qual > 0 else 0.0,
        "mean_test_origins": round(float(np.mean(test_counts)), 1) if n_qual > 0 else 0.0,
        "median_test_origins": round(float(np.median(test_counts)), 1) if n_qual > 0 else 0.0
    })

print("\n=== CHRONOLOGICAL SPLIT FEASIBILITY SUMMARY ===")
df_chrono = pd.DataFrame(chronological_feasibility)
print(df_chrono.to_string(index=False))

with open("chronological_split_feasibility.json", "w") as f:
    json.dump({
        "split_definition": {
            "panel_min_date": panel_min_date,
            "panel_max_date": panel_max_date,
            "train_end_date": train_end_dt.strftime("%Y-%m-%d"),
            "embargo_window": f"{(train_end_dt + pd.Timedelta(days=1)).strftime('%Y-%m-%d')} to {embargo_end_dt.strftime('%Y-%m-%d')}",
            "test_start_date": test_start_dt.strftime("%Y-%m-%d")
        },
        "tiers_feasibility": chronological_feasibility
    }, f, indent=2)
print("Saved chronological_split_feasibility.json")



In [ ]:
# Cell 8: Part 7 - Final Output Artifact Verification & Hashes
print("=== GENERATED ARTIFACTS VERIFICATION ===")
expected_artifacts = [
    "unmatched_excel_sample.csv",
    "unmatched_excel_summary.json",
    "diff_77_analysis.json",
    "quarantine_adjudication_summary.json",
    "target_availability_summary.json",
    "target_availability_breakdown.csv",
    "eligibility_grids.csv",
    "focus_crops_eligibility.csv",
    "chronological_split_feasibility.json"
]

manifest_outputs = []
for fname in expected_artifacts:
    if os.path.exists(fname):
        sz = os.path.getsize(fname)
        sha = compute_sha256(fname)
        line_cnt = None
        if fname.endswith(".csv"):
            with open(fname) as f:
                line_cnt = sum(1 for _ in f)
        manifest_outputs.append({
            "artifact": fname,
            "size_bytes": sz,
            "sha256": sha,
            "line_count": line_cnt
        })
        print(f"FILE: {fname:<36} | {sz:>10,} bytes | lines: {str(line_cnt):<8} | sha256: {sha}")
    else:
        print(f"MISSING: {fname}")

with open("kernel_execution_manifest.json", "w") as f:
    json.dump({"artifacts": manifest_outputs, "timestamp_utc": time.time()}, f, indent=2)
print("\nKernel Execution Completed Successfully!")

